# Pertemuan 7 — Greedy Algorithms I
Prinsip greedy, *exchange argument*, kapan greedy benar/salah. **[Modern]** greedy decoding vs beam search.

In [ ]:
import itertools, random, math
from functools import lru_cache
import numpy as np
random.seed(3)

## 1. Activity Selection — aturan "selesai paling awal"
Bandingkan tiga aturan greedy dengan solusi optimal (brute force pada instans kecil).

In [ ]:
def select(acts, key):
    acts, last, chosen = sorted(acts, key=key), -1, []
    for s, f in acts:
        if s >= last: chosen.append((s, f)); last = f
    return chosen

def optimal(acts):
    best = 0
    for r in range(len(acts) + 1):
        for sub in itertools.combinations(sorted(acts, key=lambda x: x[1]), r):
            if all(sub[i][1] <= sub[i+1][0] for i in range(len(sub) - 1)): best = max(best, r)
    return best

rules = {"selesai paling awal": lambda a: a[1], "mulai paling awal": lambda a: a[0], "durasi terpendek": lambda a: a[1] - a[0]}
wrong = {k: 0 for k in rules}
for _ in range(300):
    acts = []
    for _ in range(9):
        s = random.randint(0, 20); acts.append((s, s + random.randint(1, 8)))
    opt = optimal(acts)
    for name, key in rules.items():
        if len(select(acts, key)) < opt: wrong[name] += 1
print("Jumlah instans (dari 300) di mana aturan TIDAK optimal:", wrong)

## 2. Knapsack: fractional (greedy optimal) vs 0/1 (greedy bisa salah)

In [ ]:
def fractional(items, W):
    tot = 0
    for v, w in sorted(items, key=lambda x: x[0] / x[1], reverse=True):
        take = min(w, W); tot += v * take / w; W -= take
        if W == 0: break
    return tot
def greedy01(items, W):
    tot = 0
    for v, w in sorted(items, key=lambda x: x[0] / x[1], reverse=True):
        if w <= W: tot += v; W -= w
    return tot
def dp01(items, W):
    dp = [0] * (W + 1)
    for v, w in items:
        for c in range(W, w - 1, -1): dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]
items, W = [(60, 10), (100, 20), (120, 30)], 50      # contoh klasik CLRS
print("fractional:", fractional(items, W), "| greedy 0/1:", greedy01(items, W), "| optimal 0/1 (DP):", dp01(items, W))

## 3. Coin change: sistem koin kanonik vs non-kanonik

In [ ]:
def greedy_coins(coins, amt):
    n = 0
    for c in sorted(coins, reverse=True): n += amt // c; amt %= c
    return n
def dp_coins(coins, amt):
    INF = 10**9; dp = [0] + [INF] * amt
    for a in range(1, amt + 1): dp[a] = min((dp[a - c] + 1 for c in coins if c <= a), default=INF)
    return dp[amt]
for coins, amt in [([1, 5, 10, 25], 63), ([1, 3, 4], 6), ([1, 5, 12], 15)]:
    print(coins, "jumlah", amt, "→ greedy:", greedy_coins(coins, amt), "| optimal:", dp_coins(coins, amt))
bad = [a for a in range(1, 100) if greedy_coins([1, 3, 4], a) != dp_coins([1, 3, 4], a)]
print("Koin {1,3,4}: greedy salah untuk jumlah", bad[:10], "...")

## 4. Job scheduling — meminimalkan *maximum lateness* (Earliest Deadline First)
Exchange argument: urutan dengan *inversion* (deadline lebih besar di depan) tidak lebih baik.

In [ ]:
def max_lateness(order):
    t, worst = 0, -10**9
    for dur, dl in order: t += dur; worst = max(worst, t - dl)
    return worst
for trial in range(5):
    jobs = [(random.randint(1, 6), random.randint(3, 30)) for _ in range(7)]
    edf = max_lateness(sorted(jobs, key=lambda j: j[1]))
    opt = min(max_lateness(p) for p in itertools.permutations(jobs))
    print(f"percobaan {trial+1}: EDF={edf}  optimal(permutasi)={opt}  sama? {edf == opt}")

## 5. [Modern] Greedy decoding vs Beam search (model bahasa mainan)
Greedy memilih token terbaik **per langkah**; beam search menjaga B kandidat urutan. Greedy tidak menjamin urutan dengan probabilitas total tertinggi.

In [ ]:
LM = {"<s>": {"A": .6, "B": .4},
      "A":   {"x": .34, "y": .33, "z": .33},
      "B":   {"e": .9, "f": .1}}
def beam_search(width, steps=2):
    beams = [(0.0, ["<s>"])]
    for _ in range(steps):
        cand = []
        for lp, seq in beams:
            for tok, p in LM[seq[-1]].items(): cand.append((lp + math.log(p), seq + [tok]))
        beams = sorted(cand, reverse=True)[:width]
    return [(round(math.exp(lp), 3), seq[1:]) for lp, seq in beams]
print("greedy (B=1):", beam_search(1))
print("beam   (B=2):", beam_search(2)[:1])
print("→ greedy: langkah pertama 'A' (0.6) lebih menarik, tetapi urutan terbaik justru diawali 'B'.")

Pada LLM nyata, biaya beam search ≈ B × biaya greedy per langkah (kompleksitas waktu O(B·T·biaya_model)), tetapi *sampling* (temperature/top-p) sering dipakai untuk keragaman.

## 6. Latihan / Tugas
1. Cari **contoh tandingan** (counterexample) untuk aturan "ambil aktivitas dengan konflik paling sedikit" pada activity selection.
2. Implementasikan *interval partitioning* (jumlah ruangan minimum) secara greedy dan bandingkan dengan jumlah overlap maksimum.
3. Tambah parameter `length_penalty` pada beam search mainan dan amati perubahan hasil.
4. **Tugas:** studi kasus job scheduling — tuliskan bukti exchange argument untuk EDF dengan kata-katamu sendiri dan uji dengan 1000 instans acak.

In [ ]:
def interval_partitioning(acts):
    pass  # TODO